<a href="https://colab.research.google.com/github/filipn05/Final-Year-Project/blob/main/LogoDet3KPretrainingYOLOV8nphase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Methodology

## Phase 1: Class-Agnostic Logo Pretraining — YOLOv8n

This notebook documents **Phase 1** of the thesis methodology. The objective is to train the YOLOv8n detector to localise logos in a **class-agnostic** setting before adapting the model to the 26 target brand classes in Phase 2.

The LogoDet-3K source categories are therefore **not used as prediction classes**. Every valid logo bounding box is converted to class `0` (`logo`). The source images are shuffled with random seed `42` and divided into an 80/20 training/validation split. Ultralytics is then configured with `fraction=0.3`, so Phase 1 training uses 30% of the generated training split because of the available compute budget.

**Phase 1 configuration represented by this notebook:**
- architecture: YOLOv8n, initialised from `yolov8n.yaml`;
- task: single-class object detection (`logo`);
- input resolution: 416 × 416;
- batch size: 64;
- data-loader workers: 2;
- mosaic probability: 0.0;
- training fraction: 0.3;
- intended cumulative training budget: 40 epochs;
- execution: 5-epoch chunks with checkpoints backed up to Google Drive.

### Imports and dependencies

Imports the libraries used by the subsequent data preparation and model-training stages.


In [ ]:
import torch

# Check if CUDA (GPU support) is available
gpu_available = torch.cuda.is_available()
print(f"Is GPU available?: {gpu_available}")

if gpu_available:
    # Get the name of the GPU
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

Is GPU available?: True
GPU Device Name: Tesla T4
Number of GPUs: 1


### Google Drive setup

Mounts Google Drive so that the notebook can access persistent dataset files, training checkpoints, and outputs across Google Colab sessions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis-logodet3k'
os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


### Environment setup

Installs the Python packages required by the implementation.


In [ ]:
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print("Kaggle credentials loaded.")

# Restore from Drive if we already have a verified copy; else download fresh
if os.path.exists(f'{DRIVE_DATA_DIR}/logodet3k.zip'):
    print("\nFound existing verified zip on Drive, copy locally")
    import shutil
    shutil.copy(f'{DRIVE_DATA_DIR}/logodet3k.zip', '/content/logodet3k.zip')
else:
    print("\nNo copy on Drive yet — downloading from Kaggle")
    !pip install -q kaggle
    !kaggle datasets download -d lyly99/logodet3k

print("\nVerifying zip integrity")
!unzip -t /content/logodet3k.zip | tail -3

Kaggle credentials loaded.

No copy on Drive yet — downloading from Kaggle
Dataset URL: https://www.kaggle.com/datasets/lyly99/logodet3k
License(s): unknown
100% 2.87G/2.87G [02:37<00:00, 19.5MB/s]


Verifying zip integrity
    testing: LogoDet-3K/Transportation/zamboni/50.jpg   OK
    testing: LogoDet-3K/Transportation/zamboni/50.xml   OK
No errors detected in compressed data of /content/logodet3k.zip.


### LogoDet-3K archive preparation

Prepares the LogoDet-3K archive for use in the Colab runtime.

In [ ]:
!unzip -q /content/logodet3k.zip -d /content/logodet3k_raw
print("Extracted:", len(os.listdir('/content/logodet3k_raw')), "items")

if not os.path.exists(f'{DRIVE_DATA_DIR}/logodet3k.zip'):
    import shutil
    shutil.copy('/content/logodet3k.zip', f'{DRIVE_DATA_DIR}/logodet3k.zip')
    print("\nVerified zip backed up to Drive.")
else:
    print("\nAlready backed up to Drive.")

Extracted: 1 items

Verified zip backed up to Drive.


### Annotation conversion

Converts the source LogoDet-3K annotations into the YOLO object-detection format. The conversion preserves bounding-box localisation while representing logo instances in the format expected by the Ultralytics training pipeline.

In [ ]:
import glob
import xml.etree.ElementTree as ET
import random
import shutil
import yaml

LOCAL_RAW_DIR = '/content/logodet3k_raw'
LOCAL_YOLO_DIR = '/content/yolo_dataset'

def convert_box(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]
    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

os.makedirs(f'{LOCAL_YOLO_DIR}/images/train', exist_ok=True)
os.makedirs(f'{LOCAL_YOLO_DIR}/images/val', exist_ok=True)
os.makedirs(f'{LOCAL_YOLO_DIR}/labels/train', exist_ok=True)
os.makedirs(f'{LOCAL_YOLO_DIR}/labels/val', exist_ok=True)

image_paths = glob.glob(f'{LOCAL_RAW_DIR}/**/*.jpg', recursive=True)
classes = sorted(list(set([os.path.basename(os.path.dirname(p)) for p in image_paths])))
print(f"Successfully detected {len(classes)} distinct logo classes.")

random.seed(42)
random.shuffle(image_paths)
split_idx = int(len(image_paths) * 0.8)

Successfully detected 3000 distinct logo classes.


In [ ]:
converted_count = 0
skipped_existing = 0

for idx, img_path in enumerate(image_paths):
    split = 'train' if idx < split_idx else 'val'
    class_name = os.path.basename(os.path.dirname(img_path))
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    unique_name = f"{class_name}_{base_name}"
    xml_path = img_path.replace('.jpg', '.xml')

    out_img = f'{LOCAL_YOLO_DIR}/images/{split}/{unique_name}.jpg'
    out_lbl = f'{LOCAL_YOLO_DIR}/labels/{split}/{unique_name}.txt'

    if os.path.exists(out_img) and os.path.exists(out_lbl):
        skipped_existing += 1
        continue
    if not os.path.exists(xml_path):
        continue

    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)
        if w == 0 or h == 0:
            continue

        label_lines = []
        for obj in root.iter('object'):
            xmlbox = obj.find('bndbox')
            b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
                 float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
            bb = convert_box((w, h), b)
            label_lines.append(f"0 " + " ".join([f"{a:.6f}" for a in bb]))

        if label_lines:
            shutil.copy(img_path, out_img)
            with open(out_lbl, 'w') as f:
                f.write('\n'.join(label_lines))
            converted_count += 1
    except Exception:
        continue

with open(f'{LOCAL_YOLO_DIR}/classes.txt', 'w') as f:
    f.write('\n'.join(classes))

dataset_yaml = {'path': LOCAL_YOLO_DIR, 'train': 'images/train', 'val': 'images/val',
                 'nc': 1, 'names': {0: 'logo'}}
with open(f'{LOCAL_YOLO_DIR}/dataset.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"Newly converted: {converted_count} | Skipped (already done): {skipped_existing}")
print("images/train:", len(os.listdir(f'{LOCAL_YOLO_DIR}/images/train')))
print("images/val:", len(os.listdir(f'{LOCAL_YOLO_DIR}/images/val')))

Newly converted: 158654 | Skipped (already done): 0
images/train: 126923
images/val: 31731


In [ ]:
print("Zipping yolo_dataset")
!cd /content && zip -qr yolo_dataset_v2.zip yolo_dataset
print("Zip confirmed")

shutil.copy('/content/yolo_dataset_v2.zip', f'{DRIVE_DATA_DIR}/yolo_dataset_v2.zip')
print("yolo_dataset_v2.zip saved to Drive permanently.")

Zipping yolo_dataset
Zip confirmed
yolo_dataset_v2.zip saved to Drive permanently.


### Dataset configuration

Defines the dataset configuration consumed by Ultralytics, including the locations of the training and validation data and the class definition used during Phase 1.


In [ ]:
print("images/train:", len(os.listdir(f'{LOCAL_YOLO_DIR}/images/train')))
print("images/val:", len(os.listdir(f'{LOCAL_YOLO_DIR}/images/val')))
print("labels/train:", len(os.listdir(f'{LOCAL_YOLO_DIR}/labels/train')))
print("labels/val:", len(os.listdir(f'{LOCAL_YOLO_DIR}/labels/val')))

# dataset.yaml content
with open(f'{LOCAL_YOLO_DIR}/dataset.yaml') as f:
    print("\n" + f.read())

# Spot-check a few label files
train_labels = os.listdir(f'{LOCAL_YOLO_DIR}/labels/train')
print(f"\nSample labels (total: {len(train_labels)}):")
for fname in train_labels[:3]:
    with open(f'{LOCAL_YOLO_DIR}/labels/train/{fname}') as f:
        print(fname, '->', f.read().strip()[:150])

images/train: 126923
images/val: 31731
labels/train: 126923
labels/val: 31731

names:
  0: logo
nc: 1
path: /content/yolo_dataset
train: images/train
val: images/val


Sample labels (total: 126923):
smartwool_46.txt -> 0 0.487671 0.494253 0.876712 0.107280
0 0.754795 0.257663 0.331507 0.454023
Eegee's_107.txt -> 0 0.518775 0.438571 0.152174 0.105714
0 0.359684 0.450000 0.126482 0.071429
0 0.258893 0.462857 0.047431 0.057143
0 0.657115 0.452857 0.088933 0.06571
Caffe Ritazza_49.txt -> 0 0.441571 0.162469 0.875479 0.314861


### LogoDet-3K archive preparation

Prepares the LogoDet-3K archive for use in the Colab runtime.

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 8.4 MB/s eta 0:00:00


In [ ]:
import os
import shutil
from ultralytics import YOLO

DRIVE_DATA_DIR = '/content/drive/MyDrive/thesis-logodet3k'
CKPT_DIR = f'{DRIVE_DATA_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

RUN_NAME = 'yolov8n_class_agnostic'
LOCAL_RUN_DIR = f'runs/detect/logodet3k_pretrain/{RUN_NAME}'
LOCAL_WEIGHTS = f'{LOCAL_RUN_DIR}/weights/last.pt'
DRIVE_RUN_ZIP = f'{CKPT_DIR}/{RUN_NAME}_run.zip'
PROGRESS_FILE = f'{DRIVE_DATA_DIR}/{RUN_NAME}_epochs_done.txt'

CHUNK_SIZE = 5
TOTAL_EPOCHS = 40

# Restore checkpoint from Drive if missing locally
if not os.path.exists(LOCAL_WEIGHTS) and os.path.exists(DRIVE_RUN_ZIP):
    print("Restoring last checkpoint from Drive...")
    shutil.copy(DRIVE_RUN_ZIP, f'{RUN_NAME}_run.zip')
    !unzip -oq {RUN_NAME}_run.zip
    print("Restored.")

# Read actual progress from the tracking file
completed_epochs = 0
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        completed_epochs = int(f.read().strip())

remaining = TOTAL_EPOCHS - completed_epochs
this_chunk = min(CHUNK_SIZE, remaining)

print(f"{completed_epochs} epoch(s) already done. Training {this_chunk} more (target: {completed_epochs + this_chunk}/{TOTAL_EPOCHS}).")

if remaining <= 0:
    print(f"Already reached {TOTAL_EPOCHS} epochs — nothing more to do.")
else:
    if os.path.exists(LOCAL_WEIGHTS):
        model = YOLO(LOCAL_WEIGHTS)
        print("Continue from existing local checkpoint.")
    else:
        model = YOLO('yolov8n.yaml')
        print("Starting fresh — no local checkpoint found.")

    results = model.train(
        data=f'{LOCAL_YOLO_DIR}/dataset.yaml',
        epochs=this_chunk,
        patience=100,
        imgsz=416,
        batch=64,
        single_cls=True,
        workers=2,
        cache=False,
        mosaic=0.0,
        fraction=0.3,
        project='logodet3k_pretrain',
        name=RUN_NAME,
        exist_ok=True
    )

    completed_epochs += this_chunk
    with open(PROGRESS_FILE, 'w') as f:
        f.write(str(completed_epochs))

print("Backing up run + progress counter to Drive...")
!cd /content && zip -qr {RUN_NAME}_run.zip {LOCAL_RUN_DIR}
shutil.copy(f'{RUN_NAME}_run.zip', DRIVE_RUN_ZIP)
print(f"Backed up. {completed_epochs}/{TOTAL_EPOCHS} epochs done. Rerun this cell to continue.")

20 epoch(s) already done. Training 5 more (target: 25/40).
Continue from existing local checkpoint.
Ultralytics 8.4.154 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.3, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/d

In [ ]:
# print("Final Phase 1 checkpoint exists:", os.path.exists(f'{DRIVE_DATA_DIR}/yolov8n_pretrained_final.pt'))

Final Phase 1 checkpoint exists: True
